In [0]:
# Databricks Notebook: 04_gold_transformations
# Purpose:
# Transform Silver banking data into 10 business-ready Gold Delta tables
#
# Gold Tables:
# 1. customer_360
# 2. branch_performance
# 3. loan_portfolio
# 4. fraud_analysis
# 5. daily_transaction_trends
# 6. customer_support_metrics
# 7. monthly_financial_summary
# 8. employee_performance
# 9. cross_sell_insights
# 10. balance_distribution


# ============================================================
# SECTION 1: IMPORTS & CONFIGURATION
# ============================================================

from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
from datetime import datetime

print("=" * 80)
print("GOLD LAYER TRANSFORMATIONS - BANKING ANALYTICS")
print("=" * 80)


# Storage configuration
storage_account = "adlsbankinganalytics"

silver_base_path = (
    f"abfss://silver@{storage_account}.dfs.core.windows.net/"
)

gold_base_path = (
    f"abfss://gold@{storage_account}.dfs.core.windows.net/"
)

print(f"Silver Path : {silver_base_path}")
print(f"Gold Path   : {gold_base_path}")


# ============================================================
# SECTION 2: READ ALL SILVER TABLES
# ============================================================

print("\n" + "=" * 80)
print("READING SILVER TABLES")
print("=" * 80)


df_customers_silver = spark.read.format("delta").load(
    f"{silver_base_path}customers_silver/"
)

df_accounts_silver = spark.read.format("delta").load(
    f"{silver_base_path}accounts_silver/"
)

df_transactions_silver = spark.read.format("delta").load(
    f"{silver_base_path}transactions_silver/"
)

df_loans_silver = spark.read.format("delta").load(
    f"{silver_base_path}loans_silver/"
)

df_cards_silver = spark.read.format("delta").load(
    f"{silver_base_path}credit_cards_silver/"
)

df_branches_silver = spark.read.format("delta").load(
    f"{silver_base_path}branches_silver/"
)

df_employees_silver = spark.read.format("delta").load(
    f"{silver_base_path}employees_silver/"
)

df_fraud_silver = spark.read.format("delta").load(
    f"{silver_base_path}fraud_transactions_silver/"
)

df_insurance_silver = spark.read.format("delta").load(
    f"{silver_base_path}insurance_products_silver/"
)

df_tickets_silver = spark.read.format("delta").load(
    f"{silver_base_path}customer_support_tickets_silver/"
)

print("✓ Customers loaded")
print("✓ Accounts loaded")
print("✓ Transactions loaded")
print("✓ Loans loaded")
print("✓ Credit Cards loaded")
print("✓ Branches loaded")
print("✓ Employees loaded")
print("✓ Fraud loaded")
print("✓ Insurance loaded")
print("✓ Support Tickets loaded")

print("\n✓ ALL SILVER TABLES LOADED SUCCESSFULLY")


# ============================================================
# SECTION 3: CUSTOMER 360
# ============================================================

print("\n" + "=" * 80)
print("1. CUSTOMER 360")
print("=" * 80)


# Account-level aggregation
account_agg = (
    df_accounts_silver
    .groupBy("customer_id")
    .agg(
        count("account_id").alias("total_accounts"),

        sum(
            when(col("is_active"), 1).otherwise(0)
        ).alias("active_accounts"),

        sum("balance").alias("total_balance"),

        avg("balance").alias("avg_balance")
    )
)


# Loan-level aggregation
loan_agg = (
    df_loans_silver
    .groupBy("customer_id")
    .agg(
        count("loan_id").alias("total_loans"),

        sum(
            when(col("is_active_loan"), 1).otherwise(0)
        ).alias("active_loans"),

        sum("loan_amount").alias("total_loan_amount"),

        sum(
            when(col("is_defaulted"), 1).otherwise(0)
        ).alias("defaulted_loans")
    )
)


# Credit-card aggregation
card_agg = (
    df_cards_silver
    .groupBy("customer_id")
    .agg(
        count("card_id").alias("total_cards"),

        sum(
            when(col("is_active_card"), 1).otherwise(0)
        ).alias("active_cards"),

        sum("credit_limit").alias("total_credit_limit"),

        sum("outstanding_balance").alias(
            "total_card_outstanding"
        ),

        avg("credit_utilization_pct").alias(
            "avg_credit_utilization"
        )
    )
)


# Insurance aggregation
insurance_agg = (
    df_insurance_silver
    .groupBy("customer_id")
    .agg(
        count("policy_id").alias("total_policies"),

        sum(
            when(col("is_active_policy"), 1).otherwise(0)
        ).alias("active_policies"),

        sum("premium_amount").alias("total_premium")
    )
)


# Support aggregation
support_agg = (
    df_tickets_silver
    .groupBy("customer_id")
    .agg(
        count("ticket_id").alias("total_tickets"),

        sum(
            when(col("is_resolved"), 1).otherwise(0)
        ).alias("resolved_tickets"),

        avg("resolution_time_days").alias(
            "avg_resolution_days"
        )
    )
)


# Build Customer 360
df_customer_360 = (
    df_customers_silver

    .join(account_agg, "customer_id", "left")

    .join(loan_agg, "customer_id", "left")

    .join(card_agg, "customer_id", "left")

    .join(insurance_agg, "customer_id", "left")

    .join(support_agg, "customer_id", "left")

    .fillna(0)

    .withColumn(
        "total_assets",
        col("total_balance")
        + col("total_credit_limit")
        - col("total_card_outstanding")
    )

    .withColumn(
        "risk_score",
        when(col("defaulted_loans") > 0, "High")
        .when(col("avg_credit_utilization") > 80, "Medium-High")
        .when(col("avg_credit_utilization") > 50, "Medium")
        .otherwise("Low")
    )
)


gold_customer360_path = f"{gold_base_path}customer_360/"

(
    df_customer_360
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(gold_customer360_path)
)

print(
    f"✓ Customer 360 saved: {gold_customer360_path}"
)

display(df_customer_360.limit(10))

'''
# ============================================================
# SECTION 4: BRANCH PERFORMANCE
# ============================================================

print("\n" + "=" * 80)
print("2. BRANCH PERFORMANCE")
print("=" * 80)


branch_accounts = (
    df_accounts_silver
    .groupBy("branch_id")
    .agg(
        count("account_id").alias("total_accounts"),
        sum("balance").alias("total_deposits"),
        avg("balance").alias("avg_deposit")
    )
)


branch_employees = (
    df_employees_silver
    .filter(col("is_active_employee"))
    .groupBy("branch_id")
    .agg(
        count("employee_id").alias("total_employees"),
        avg("salary").alias("avg_salary")
    )
)


branch_loans = (
    df_loans_silver
    .groupBy("branch_id")
    .agg(
        count("loan_id").alias("total_loans"),
        sum("loan_amount").alias("total_loan_disbursed"),
        avg("interest_rate").alias("avg_interest_rate")
    )
)


# NOTE:
# This requires branch_id to exist in transactions_silver.
# If your transaction table does not contain branch_id,
# remove the transaction section from branch performance.

if "branch_id" in df_transactions_silver.columns:

    branch_transactions = (
        df_transactions_silver
        .groupBy("branch_id")
        .agg(
            count("transaction_id").alias(
                "total_transactions"
            ),

            sum(
                when(
                    col("transaction_type") == "Debit",
                    col("amount")
                ).otherwise(0)
            ).alias("total_debits"),

            sum(
                when(
                    col("transaction_type") == "Credit",
                    col("amount")
                ).otherwise(0)
            ).alias("total_credits")
        )
    )

else:

    branch_transactions = None


df_branch_performance = (
    df_branches_silver

    .join(
        branch_accounts,
        "branch_id",
        "left"
    )

    .join(
        branch_employees,
        "branch_id",
        "left"
    )

    .join(
        branch_loans,
        "branch_id",
        "left"
    )
)


if branch_transactions is not None:

    df_branch_performance = (
        df_branch_performance
        .join(
            branch_transactions,
            "branch_id",
            "left"
        )
    )

else:

    df_branch_performance = (
        df_branch_performance
        .withColumn(
            "total_transactions",
            lit(0)
        )
        .withColumn(
            "total_debits",
            lit(0)
        )
        .withColumn(
            "total_credits",
            lit(0)
        )
    )


df_branch_performance = (
    df_branch_performance

    .fillna(0)

    .withColumn(
        "net_flow",
        col("total_credits")
        - col("total_debits")
    )

    .withColumn(
        "employee_efficiency",
        when(
            col("total_employees") > 0,
            round(
                col("total_accounts")
                / col("total_employees"),
                2
            )
        ).otherwise(0)
    )

    .select(
        "branch_id",
        "branch_name_clean",
        "city_clean",
        "state_clean",
        "total_accounts",
        "total_deposits",
        "avg_deposit",
        "total_employees",
        "avg_salary",
        "employee_efficiency",
        "total_loans",
        "total_loan_disbursed",
        "avg_interest_rate",
        "total_transactions",
        "total_debits",
        "total_credits",
        "net_flow"
    )
)


gold_branch_path = (
    f"{gold_base_path}branch_performance/"
)

(
    df_branch_performance
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(gold_branch_path)
)

print(
    f"✓ Branch Performance saved: {gold_branch_path}"
)

display(df_branch_performance.limit(10))
'''

# ============================================================
# SECTION 5: LOAN PORTFOLIO
# ============================================================

print("\n" + "=" * 80)
print("3. LOAN PORTFOLIO")
print("=" * 80)


df_loan_portfolio = (
    df_loans_silver

    .join(
        df_customers_silver.select(
            "customer_id",
            "age_group",
            "income_bracket",
            "city"
        ),
        "customer_id",
        "left"
    )

    .groupBy(
        "loan_type",
        "loan_status_standardized",
        "age_group",
        "income_bracket"
    )

    .agg(
        count("loan_id").alias(
            "loan_count"
        ),

        sum("loan_amount").alias(
            "total_loan_amount"
        ),

        avg("loan_amount").alias(
            "avg_loan_amount"
        ),

        avg("interest_rate").alias(
            "avg_interest_rate"
        ),

        sum("total_interest").alias(
            "total_interest_income"
        ),

        sum("total_payable").alias(
            "total_receivable"
        )
    )

    .orderBy(
        col("total_loan_amount").desc()
    )
)


gold_loan_path = (
    f"{gold_base_path}loan_portfolio/"
)

(
    df_loan_portfolio
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(gold_loan_path)
)

print(
    f"✓ Loan Portfolio saved: {gold_loan_path}"
)

display(df_loan_portfolio.limit(10))


# ============================================================
# SECTION 6: FRAUD ANALYSIS
# ============================================================

print("\n" + "=" * 80)
print("4. FRAUD ANALYSIS")
print("=" * 80)


df_fraud_analysis = (
    df_fraud_silver

    .join(
        df_transactions_silver.select(
            "transaction_id",
            "transaction_date",
            "amount",
            "payment_mode_standardized",
            "merchant_name"
        ),
        "transaction_id",
        "left"
    )

    .withColumn(
        "fraud_year",
        year("detected_date")
    )

    .withColumn(
        "fraud_month",
        month("detected_date")
    )

    .groupBy(
        "fraud_year",
        "fraud_month",
        "fraud_type",
        "risk_level"
    )

    .agg(
        count("fraud_id").alias(
            "fraud_count"
        ),

        sum("loss_amount").alias(
            "total_loss"
        ),

        avg("loss_amount").alias(
            "avg_loss"
        ),

        sum(
            when(
                col("is_confirmed_fraud"),
                1
            ).otherwise(0)
        ).alias(
            "confirmed_frauds"
        ),

        sum(
            when(
                col("is_false_positive"),
                1
            ).otherwise(0)
        ).alias(
            "false_positives"
        )
    )

    .withColumn(
        "confirmation_rate",
        when(
            col("fraud_count") > 0,
            round(
                col("confirmed_frauds")
                / col("fraud_count") * 100,
                2
            )
        ).otherwise(0)
    )

    .orderBy(
        "fraud_year",
        "fraud_month"
    )
)


gold_fraud_path = (
    f"{gold_base_path}fraud_analysis/"
)

(
    df_fraud_analysis
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(gold_fraud_path)
)

print(
    f"✓ Fraud Analysis saved: {gold_fraud_path}"
)

display(df_fraud_analysis.limit(10))


# ============================================================
# SECTION 7: DAILY TRANSACTION TRENDS
# ============================================================

print("\n" + "=" * 80)
print("5. DAILY TRANSACTION TRENDS")
print("=" * 80)


df_daily_transaction_trends = (
    df_transactions_silver

    .groupBy(
        "transaction_date",
        "transaction_type",
        "payment_mode_standardized"
    )

    .agg(
        count("transaction_id").alias(
            "transaction_count"
        ),

        sum("amount").alias(
            "total_amount"
        ),

        avg("amount").alias(
            "avg_transaction_amount"
        ),

        sum(
            when(
                col("is_success"),
                1
            ).otherwise(0)
        ).alias(
            "successful_count"
        ),

        sum(
            when(
                col("is_failed"),
                1
            ).otherwise(0)
        ).alias(
            "failed_count"
        )
    )

    .withColumn(
        "success_rate",
        when(
            col("transaction_count") > 0,
            round(
                col("successful_count")
                / col("transaction_count") * 100,
                2
            )
        ).otherwise(0)
    )

    .orderBy("transaction_date")
)


gold_daily_path = (
    f"{gold_base_path}daily_transaction_trends/"
)

(
    df_daily_transaction_trends
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(gold_daily_path)
)

print(
    f"✓ Daily Transaction Trends saved: {gold_daily_path}"
)

display(
    df_daily_transaction_trends.limit(10)
)


# ============================================================
# SECTION 8: CUSTOMER SUPPORT METRICS
# ============================================================

print("\n" + "=" * 80)
print("6. CUSTOMER SUPPORT METRICS")
print("=" * 80)


df_customer_support_metrics = (
    df_tickets_silver

    .groupBy(
        "issue_type_standardized",
        "priority",
        "channel_standardized"
    )

    .agg(
        count("ticket_id").alias(
            "total_tickets"
        ),

        sum(
            when(
                col("is_resolved"),
                1
            ).otherwise(0)
        ).alias(
            "resolved_tickets"
        ),

        sum(
            when(
                col("is_open"),
                1
            ).otherwise(0)
        ).alias(
            "open_tickets"
        ),

        avg(
            "resolution_time_days"
        ).alias(
            "avg_resolution_days"
        ),

        min(
            "resolution_time_days"
        ).alias(
            "min_resolution_days"
        ),

        max(
            "resolution_time_days"
        ).alias(
            "max_resolution_days"
        )
    )

    .withColumn(
        "resolution_rate",
        when(
            col("total_tickets") > 0,
            round(
                col("resolved_tickets")
                / col("total_tickets") * 100,
                2
            )
        ).otherwise(0)
    )

    .orderBy(
        col("total_tickets").desc()
    )
)


gold_support_path = (
    f"{gold_base_path}customer_support_metrics/"
)

(
    df_customer_support_metrics
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(gold_support_path)
)

print(
    f"✓ Customer Support Metrics saved: "
    f"{gold_support_path}"
)

display(
    df_customer_support_metrics.limit(10)
)


# ============================================================
# SECTION 9: MONTHLY FINANCIAL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("7. MONTHLY FINANCIAL SUMMARY")
print("=" * 80)


df_monthly_financial_summary = (
    df_transactions_silver

    .withColumn(
        "transaction_year",
        year("transaction_date")
    )

    .withColumn(
        "transaction_month",
        month("transaction_date")
    )

    .groupBy(
        "transaction_year",
        "transaction_month"
    )

    .agg(
        count("transaction_id").alias(
            "total_transactions"
        ),

        sum(
            when(
                col("transaction_type") == "Credit",
                col("amount")
            ).otherwise(0)
        ).alias(
            "total_credits"
        ),

        sum(
            when(
                col("transaction_type") == "Debit",
                col("amount")
            ).otherwise(0)
        ).alias(
            "total_debits"
        ),

        sum(
            when(
                col("payment_mode_standardized") == "UPI",
                col("amount")
            ).otherwise(0)
        ).alias(
            "upi_volume"
        ),

        sum(
            when(
                col("payment_mode_standardized") == "Net Banking",
                col("amount")
            ).otherwise(0)
        ).alias(
            "netbanking_volume"
        ),

        sum(
            when(
                col("payment_mode_standardized") == "RTGS",
                col("amount")
            ).otherwise(0)
        ).alias(
            "rtgs_volume"
        ),

        sum(
            when(
                col("payment_mode_standardized") == "IMPS",
                col("amount")
            ).otherwise(0)
        ).alias(
            "imps_volume"
        )
    )

    .withColumn(
        "net_flow",
        col("total_credits")
        - col("total_debits")
    )

    .orderBy(
        "transaction_year",
        "transaction_month"
    )
)


gold_monthly_path = (
    f"{gold_base_path}monthly_financial_summary/"
)

(
    df_monthly_financial_summary
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(gold_monthly_path)
)

print(
    f"✓ Monthly Financial Summary saved: "
    f"{gold_monthly_path}"
)

display(
    df_monthly_financial_summary.limit(12)
)


# ============================================================
# SECTION 10: EMPLOYEE PERFORMANCE
# ============================================================

print("\n" + "=" * 80)
print("8. EMPLOYEE PERFORMANCE")
print("=" * 80)


# The Silver documentation creates tenure_years.
# We derive experience_level here for Gold reporting.

df_employees_gold_base = (
    df_employees_silver

    .withColumn(
        "experience_level",
        when(
            col("tenure_years") < 2,
            "Junior (<2 years)"
        )
        .when(
            col("tenure_years") < 5,
            "Mid-Level (2-5 years)"
        )
        .when(
            col("tenure_years") < 10,
            "Senior (5-10 years)"
        )
        .otherwise(
            "Expert (>10 years)"
        )
    )
)


df_employee_performance = (
    df_employees_gold_base

    .join(
        df_branches_silver.select(
            "branch_id",
            "branch_name_clean",
            "city_clean"
        ),
        "branch_id",
        "left"
    )

    .groupBy(
        "branch_name_clean",
        "designation_clean",
        "experience_level"
    )

    .agg(
        count("employee_id").alias(
            "employee_count"
        ),

        avg("salary").alias(
            "avg_salary"
        ),

        sum(
            when(
                col("is_active_employee"),
                1
            ).otherwise(0)
        ).alias(
            "active_employees"
        )
    )

    .orderBy(
        col("avg_salary").desc()
    )
)


gold_employee_path = (
    f"{gold_base_path}employee_performance/"
)

(
    df_employee_performance
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(gold_employee_path)
)

print(
    f"✓ Employee Performance saved: "
    f"{gold_employee_path}"
)

display(
    df_employee_performance.limit(10)
)


# ============================================================
# SECTION 11: CROSS-SELL INSIGHTS
# ============================================================

print("\n" + "=" * 80)
print("9. CROSS-SELL INSIGHTS")
print("=" * 80)


# Account products per customer
customer_accounts = (
    df_accounts_silver
    .groupBy("customer_id")
    .agg(
        count("account_id").alias(
            "total_accounts"
        ),

        collect_set(
            "account_type"
        ).alias(
            "account_types"
        )
    )
)


# Loans per customer
customer_loans = (
    df_loans_silver
    .groupBy("customer_id")
    .agg(
        count("loan_id").alias(
            "has_loan"
        )
    )
)


# Cards per customer
customer_cards = (
    df_cards_silver
    .groupBy("customer_id")
    .agg(
        count("card_id").alias(
            "has_card"
        )
    )
)


# Insurance per customer
customer_insurance = (
    df_insurance_silver
    .groupBy("customer_id")
    .agg(
        count("policy_id").alias(
            "has_insurance"
        )
    )
)


df_cross_sell = (
    df_customers_silver

    .join(
        customer_accounts,
        "customer_id",
        "left"
    )

    .join(
        customer_loans,
        "customer_id",
        "left"
    )

    .join(
        customer_cards,
        "customer_id",
        "left"
    )

    .join(
        customer_insurance,
        "customer_id",
        "left"
    )

    .fillna({
        "total_accounts": 0,
        "has_loan": 0,
        "has_card": 0,
        "has_insurance": 0
    })

    .withColumn(
        "product_count",
        col("total_accounts")
        + col("has_loan")
        + col("has_card")
        + col("has_insurance")
    )

    .groupBy(
        "age_group",
        "income_bracket",
        "city"
    )

    .agg(
        count("customer_id").alias(
            "customer_count"
        ),

        avg("product_count").alias(
            "avg_products_per_customer"
        ),

        sum(
            when(
                col("has_loan") > 0,
                1
            ).otherwise(0)
        ).alias(
            "loan_customers"
        ),

        sum(
            when(
                col("has_card") > 0,
                1
            ).otherwise(0)
        ).alias(
            "card_customers"
        ),

        sum(
            when(
                col("has_insurance") > 0,
                1
            ).otherwise(0)
        ).alias(
            "insurance_customers"
        )
    )

    .withColumn(
        "loan_penetration",
        when(
            col("customer_count") > 0,
            round(
                col("loan_customers")
                / col("customer_count") * 100,
                2
            )
        ).otherwise(0)
    )

    .withColumn(
        "card_penetration",
        when(
            col("customer_count") > 0,
            round(
                col("card_customers")
                / col("customer_count") * 100,
                2
            )
        ).otherwise(0)
    )

    .withColumn(
        "insurance_penetration",
        when(
            col("customer_count") > 0,
            round(
                col("insurance_customers")
                / col("customer_count") * 100,
                2
            )
        ).otherwise(0)
    )
)


gold_cross_sell_path = (
    f"{gold_base_path}cross_sell_insights/"
)

(
    df_cross_sell
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(gold_cross_sell_path)
)

print(
    f"✓ Cross-Sell Insights saved: "
    f"{gold_cross_sell_path}"
)

display(df_cross_sell.limit(10))


# ============================================================
# SECTION 12: BALANCE DISTRIBUTION
# ============================================================

print("\n" + "=" * 80)
print("10. BALANCE DISTRIBUTION")
print("=" * 80)


df_balance_distribution = (
    df_accounts_silver

    .join(
        df_branches_silver.select(
            "branch_id",
            "city_clean",
            "state_clean"
        ),
        "branch_id",
        "left"
    )

    .groupBy(
        "account_type",
        "balance_category",
        "city_clean",
        "state_clean"
    )

    .agg(
        count("account_id").alias(
            "account_count"
        ),

        sum("balance").alias(
            "total_balance"
        ),

        avg("balance").alias(
            "avg_balance"
        ),

        sum(
            when(
                col("is_active"),
                1
            ).otherwise(0)
        ).alias(
            "active_accounts"
        )
    )

    .orderBy(
        col("total_balance").desc()
    )
)


gold_balance_path = (
    f"{gold_base_path}balance_distribution/"
)

(
    df_balance_distribution
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(gold_balance_path)
)

print(
    f"✓ Balance Distribution saved: "
    f"{gold_balance_path}"
)

display(
    df_balance_distribution.limit(10)
)


# ============================================================
# SECTION 13: GOLD LAYER SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("GOLD LAYER COMPLETED SUCCESSFULLY")
print("=" * 80)

gold_tables = [
    ("customer_360", gold_customer360_path),
    ("branch_performance", gold_branch_path),
    ("loan_portfolio", gold_loan_path),
    ("fraud_analysis", gold_fraud_path),
    ("daily_transaction_trends", gold_daily_path),
    ("customer_support_metrics", gold_support_path),
    ("monthly_financial_summary", gold_monthly_path),
    ("employee_performance", gold_employee_path),
    ("cross_sell_insights", gold_cross_sell_path),
    ("balance_distribution", gold_balance_path)
]

for table_name, table_path in gold_tables:
    print(f"✓ {table_name:<30} -> {table_path}")

print("\n" + "=" * 80)
print("10 GOLD DASHBOARDS / TABLES CREATED")
print("=" * 80)

### Load Tables in Gold Schema

In [0]:
print(storage_account)

In [0]:
%sql
-- Gold Tables 
CREATE TABLE IF NOT EXISTS banking_catalog.gold.gold_customer_360 
USING DELTA LOCATION 'abfss://gold@adlsbankinganalytics.dfs.core.windows.net/customer_360/'; 

CREATE TABLE IF NOT EXISTS banking_catalog.gold.gold_branch_performance 
USING DELTA LOCATION 'abfss://gold@adlsbankinganalytics.dfs.core.windows.net/branch_performance/';

CREATE TABLE IF NOT EXISTS banking_catalog.gold.gold_loan_portfolio 
USING DELTA LOCATION 'abfss://gold@adlsbankinganalytics.dfs.core.windows.net/loan_portfolio/'; 

CREATE TABLE IF NOT EXISTS banking_catalog.gold.gold_fraud_analysis 
USING DELTA LOCATION 'abfss://gold@adlsbankinganalytics.dfs.core.windows.net/fraud_analysis/'; 

CREATE TABLE IF NOT EXISTS banking_catalog.gold.gold_daily_transaction_trends 
USING DELTA LOCATION 'abfss://gold@adlsbankinganalytics.dfs.core.windows.net/daily_transaction_trends/'; 

CREATE TABLE IF NOT EXISTS banking_catalog.gold.gold_customer_support_metrics 
USING DELTA LOCATION 'abfss://gold@adlsbankinganalytics.dfs.core.windows.net/customer_support_metrics/'; 

CREATE TABLE IF NOT EXISTS banking_catalog.gold.gold_monthly_financial_summary 
USING DELTA LOCATION 'abfss://gold@adlsbankinganalytics.dfs.core.windows.net/monthly_financial_summary/'; 

CREATE TABLE IF NOT EXISTS banking_catalog.gold.gold_employee_performance 
USING DELTA LOCATION 'abfss://gold@adlsbankinganalytics.dfs.core.windows.net/employee_performance/'; 

CREATE TABLE IF NOT EXISTS banking_catalog.gold.gold_cross_sell_insights 
USING DELTA LOCATION 'abfss://gold@adlsbankinganalytics.dfs.core.windows.net/cross_sell_insights/'; 

CREATE TABLE IF NOT EXISTS banking_catalog.gold.gold_balance_distribution 
USING DELTA LOCATION 'abfss://gold@adlsbankinganalytics.dfs.core.windows.net/balance_distribution/';